# Noetebook Mentoría: Proyecto Final

## Pipline del proyecto

1. EDA
* distribuciones (log)
* outliers
* correlaciones (Spearman)
2. Feature engineering
* ratios
* logs
* flags
3. Tests
* Mann-Whitney → corto vs largo
* Kruskal → género
* Spearman → relaciones
4. Modelo
* regresión (explicativo)
* clasificación (is_hit)

## Información del proyecto

| Nombre      | Significado               |
| ----------- | ------------------------- |
| df          | raw data                  |
| df_tracks   | datos de tracks |
| df_raw    | datos descargados             |
| df_clean    | datos limpios             |
| df_analysis | datos para análisis       |
| df_enriched | datos con features nuevas |

* tag_list[ ] = Lista de tracks por género (tag).
* tracks [ ] = Lista de tracks según tags encontrados.


## Imports

In [2]:
import os
import time
import warnings
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns


In [8]:
import pandas
print(pandas.__file__)

AttributeError: partially initialized module 'pandas' has no attribute 'plotting' (most likely due to a circular import)

## Peticiones API 

In [ ]:
API_KEY = '63e059c3c912a3f642daf2372484d183'
BASE_URL = f'http://ws.audioscrobbler.com/2.0/?api_key={API_KEY}&'

| Endpoint | Qué devuelve | Para qué lo usamos |
|---|---|---|
| `tag.getTopTags` | Tags (géneros) de cada track |  Enriquece genre_tag |
| `track.getInfo` |  |  |

### Get top tags

In [ ]:
re = requests.get(f'{BASE_URL}method=tag.getTopTags&format=json')
re_json = re.json()

In [ ]:
re_json['toptags']['tag'][:10]

[{'name': 'rock', 'count': 4069751, 'reach': 402947},
 {'name': 'electronic', 'count': 2499329, 'reach': 262248},
 {'name': 'seen live', 'count': 2194811, 'reach': 82562},
 {'name': 'alternative', 'count': 2131086, 'reach': 267224},
 {'name': 'pop', 'count': 2082944, 'reach': 233788},
 {'name': 'indie', 'count': 2065519, 'reach': 260476},
 {'name': 'female vocalists', 'count': 1634597, 'reach': 169166},
 {'name': 'metal', 'count': 1305855, 'reach': 159027},
 {'name': 'alternative rock', 'count': 1230526, 'reach': 170271},
 {'name': 'jazz', 'count': 1196690, 'reach': 149995}]

In [ ]:
tag_list = []
for i in re_json['toptags']['tag']:
  tag_list.append(i['name'])
len(tag_list)

50

DUDA: no estoy guardando la info de el diccionario entero de tags verdad? entonces no estoy descargando count y reach?
Si quiero obtener esa info 'count' y 'reach'tengoque añadir otro bucle creando otra lista? tendria que volver a descargarme los datos otra vezy esperar 3 horas?

### top tracks for tag

In [8]:
#Prueba de salida
prueba = requests.get(f'{BASE_URL}method=tag.gettoptracks&tag=rock&format=json&limit=1000')
prueba_json = prueba.json()
prueba_json

{'tracks': {'track': [{'name': 'The Chain - 2004 Remaster',
    'duration': '269',
    'mbid': 'cd06c484-9319-4376-a104-504871e19756',
    'url': 'https://www.last.fm/music/Fleetwood+Mac/_/The+Chain+-+2004+Remaster',
    'streamable': {'#text': '0', 'fulltrack': '0'},
    'artist': {'name': 'Fleetwood Mac',
     'mbid': 'bd13909f-1c29-4c27-a874-d4aaf27c5b1a',
     'url': 'https://www.last.fm/music/Fleetwood+Mac'},
    'image': [{'#text': 'https://lastfm.freetls.fastly.net/i/u/34s/2a96cbd8b46e442fc41c2b86b821562f.png',
      'size': 'small'},
     {'#text': 'https://lastfm.freetls.fastly.net/i/u/64s/2a96cbd8b46e442fc41c2b86b821562f.png',
      'size': 'medium'},
     {'#text': 'https://lastfm.freetls.fastly.net/i/u/174s/2a96cbd8b46e442fc41c2b86b821562f.png',
      'size': 'large'},
     {'#text': 'https://lastfm.freetls.fastly.net/i/u/300x300/2a96cbd8b46e442fc41c2b86b821562f.png',
      'size': 'extralarge'}],
    '@attr': {'rank': '1'}},
   {'name': 'Iris',
    'duration': '294',
    '

In [9]:
pd.DataFrame(prueba_json.get('tracks', {}).get('track', []))[['name', 'duration', 'mbid', 'artist']].info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   name      1000 non-null   str   
 1   duration  1000 non-null   str   
 2   mbid      983 non-null    str   
 3   artist    1000 non-null   object
dtypes: object(1), str(3)
memory usage: 31.4+ KB


In [10]:
tracks = []

for tag in tag_list:
  data = requests.get(f'{BASE_URL}method=tag.gettoptracks&tag={tag}&format=json&limit=1000')
  data_json = data.json()
  tag_tracks = data_json.get('tracks', {}).get('track', [])
  for t in tag_tracks:
            artist_name = t.get('artist', {}).get('name', '')
            tracks.append({
                'name'       : t.get('name', ''),
                'artist'     : artist_name,
                'mbid'       : t.get('mbid', np.nan)})
tracks

[{'name': 'The Chain - 2004 Remaster',
  'artist': 'Fleetwood Mac',
  'mbid': 'cd06c484-9319-4376-a104-504871e19756'},
 {'name': 'Iris',
  'artist': 'Goo Goo Dolls',
  'mbid': '02a75d1a-977b-4430-9db4-5d02d26e1d85'},
 {'name': 'Everlong',
  'artist': 'Foo Fighters',
  'mbid': '00779e5b-e581-3fcb-b0af-d6150e446b23'},
 {'name': "Lover, You Should've Come Over",
  'artist': 'Jeff Buckley',
  'mbid': '027cdcbb-96e7-351b-a7b5-d1004440e0f1'},
 {'name': 'Still Into You',
  'artist': 'Paramore',
  'mbid': '024d0cca-7f37-3456-9b42-1e56c4f0d460'},
 {'name': 'Bring Me to Life',
  'artist': 'Evanescence',
  'mbid': '00586963-0545-3a1e-8fa9-d21ebfbdb9f6'},
 {'name': 'Mr. Brightside',
  'artist': 'The Killers',
  'mbid': '0157fd48-a126-4af1-ad1f-4fbb37911b90'},
 {'name': 'All I Wanted',
  'artist': 'Paramore',
  'mbid': '013c6618-aac8-335c-b45f-2b39ed6f47ae'},
 {'name': 'Scar Tissue',
  'artist': 'Red Hot Chili Peppers',
  'mbid': '01de644b-448d-4aac-ba09-5c6bdc1bd8c1'},
 {'name': "Can't Stop",
  'a

In [11]:
df_tracks = pd.DataFrame(tracks)
df_tracks

,name,artist,mbid
0,The Chain - 2004 Remaster,Fleetwood Mac,cd06c484-9319-4376-a104-504871e19756
1,Iris,Goo Goo Dolls,02a75d1a-977b-4430-9db4-5d02d26e1d85
2,Everlong,Foo Fighters,00779e5b-e581-3fcb-b0af-d6150e446b23
3,"Lover, You Should've Come Over",Jeff Buckley,027cdcbb-96e7-351b-a7b5-d1004440e0f1
4,Still Into You,Paramore,024d0cca-7f37-3456-9b42-1e56c4f0d460
...,...,...,...
48129,Along,Alexander Kowalski,76feb08b-5ef5-3d32-a4d7-e6a23579ddc1
48130,The Chains of Babylon,Johannes Heil,00b5c482-d322-3e8c-9d61-e5b6083d55cc
48131,Phonky Tribu,Funk Tribu,db11568a-d79e-4af7-959b-95bc703e607a
48132,PHD,The Crystal Method,18dfa35b-028b-3e02-be24-c12d7f4c40c6


In [12]:
df_tracks.info()

<class 'pandas.DataFrame'>
RangeIndex: 48134 entries, 0 to 48133
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   name    48134 non-null  str  
 1   artist  48134 non-null  str  
 2   mbid    46974 non-null  str  
dtypes: str(3)
memory usage: 1.1 MB


In [13]:
df_tracks.duplicated().sum()

np.int64(12751)

In [14]:
df_tracks[df_tracks.duplicated()]['mbid'].isna().sum()

np.int64(194)

>Revisar duplicados y mbid nulos

In [15]:
#Resolución rápida
df_tracks.drop_duplicates(inplace=True)
df_tracks.dropna(inplace=True)
df_tracks.info()

<class 'pandas.DataFrame'>
Index: 34417 entries, 0 to 48133
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   name    34417 non-null  str  
 1   artist  34417 non-null  str  
 2   mbid    34417 non-null  str  
dtypes: str(3)
memory usage: 1.1 MB


### Get info

In [ ]:
prueba = requests.get(f'{BASE_URL}method=track.getInfo&mbid={df_tracks['mbid'].iloc[0]}&format=json')
prueba_json = prueba.json()
prueba_json


SyntaxError: f-string: unmatched '[' (1947139580.py, line 1)

In [ ]:
prueba_json.get('track')

{'name': 'The Chain',
 'mbid': '021c73b9-c6d5-41de-a852-8b990b7e24e7',
 'url': 'https://www.last.fm/music/Fleetwood+Mac/_/The+Chain',
 'duration': '269000',
 'streamable': {'#text': '0', 'fulltrack': '0'},
 'listeners': '624117',
 'playcount': '4514996',
 'artist': {'name': 'Fleetwood Mac',
  'mbid': 'bd13909f-1c29-4c27-a874-d4aaf27c5b1a',
  'url': 'https://www.last.fm/music/Fleetwood+Mac'},
 'album': {'artist': 'Fleetwood Mac',
  'title': 'Rumours',
  'url': 'https://www.last.fm/music/Fleetwood+Mac/Rumours',
  'image': [{'#text': 'https://lastfm.freetls.fastly.net/i/u/34s/349d64820e124b77cb5275ab03042693.png',
    'size': 'small'},
   {'#text': 'https://lastfm.freetls.fastly.net/i/u/64s/349d64820e124b77cb5275ab03042693.png',
    'size': 'medium'},
   {'#text': 'https://lastfm.freetls.fastly.net/i/u/174s/349d64820e124b77cb5275ab03042693.png',
    'size': 'large'},
   {'#text': 'https://lastfm.freetls.fastly.net/i/u/300x300/349d64820e124b77cb5275ab03042693.png',
    'size': 'extralarge'

In [ ]:
tracks_info = []

for mbid in df_tracks['mbid']:
  time.sleep(0.1) # Add a small delay to avoid hitting API rate limits
  data = requests.get(f'{BASE_URL}method=track.getInfo&mbid={mbid}&format=json')

  # Check for successful HTTP status code
  if data.status_code == 200:
    try:
      data_json = data.json()
      t = data_json.get('track')
      if t:
        artist_name = t.get('artist', {}).get('name', np.nan)
        tracks_info.append({
          'name'       : t.get('name', np.nan),
          'artist'     : artist_name,
          'duration'   : t.get('duration', np.nan),
          'mbid'       : t.get('mbid', np.nan),
          'tag'        : t.get('toptags', {}).get('tag', np.nan),
          'streamable' : t.get('streamable', {}).get('fulltrack', np.nan),
          'listeners'  : t.get('listeners', np.nan),
          'playcount'  : t.get('playcount', np.nan),
          'published'  : t.get('wiki', {}).get('published', np.nan)
          })
      else:
        # Handle cases where 'track' key is missing in a valid JSON response (e.g., API returns {'error': ..., 'message': 'Track not found'}) or is None
        tracks_info.append({
            'name': np.nan,
            'artist': np.nan,
            'duration': np.nan,
            'mbid': mbid,
            'tag': np.nan,
            'streamable': np.nan,
            'listeners': np.nan,
            'playcount': np.nan,
            'published': np.nan
        })
    except requests.exceptions.JSONDecodeError:
      # Handle cases where the response is not valid JSON, even if status code was 200
      print(f"Warning: JSONDecodeError for mbid: {mbid}. Response was not valid JSON. Status code: {data.status_code}. Content start: {data.text[:100]}...")
      tracks_info.append({
          'name': np.nan,
          'artist': np.nan,
          'duration': np.nan,
          'mbid': mbid,
          'tag': np.nan,
          'streamable': np.nan,
          'listeners': np.nan,
          'playcount': np.nan,
          'published': np.nan
      })
  else:
    # Handle cases where the HTTP request itself failed (non-200 status code)
    print(f"Error: API request failed for mbid: {mbid} with status code {data.status_code}. Content start: {data.text[:100]}...")
    tracks_info.append({
        'name': np.nan,
        'artist': np.nan,
        'duration': np.nan,
        'mbid': mbid,
        'tag': np.nan,
        'streamable': np.nan,
        'listeners': np.nan,
        'playcount': np.nan,
        'published': np.nan
    })

df_tracks_info = pd.DataFrame(tracks_info)

Error: API request failed for mbid: 08aef7f3-34f7-4db9-80f4-7e79b0390f31 with status code 500. Content start: {"message":"Operation failed - Most likely the backend service failed. Please try again.","error":8}...


In [ ]:
pd.DataFrame(tracks_info).to_csv("backup_tracks.csv", index=False)

NameError: name 'tracks_info' is not defined

In [ ]:
pd.DataFrame(df_tracks_info)

,name,artist,duration,mbid,tag,streamable,listeners,playcount,published
0,The Chain,Fleetwood Mac,269000,021c73b9-c6d5-41de-a852-8b990b7e24e7,[],0,624117,4514996,"19 Dec 2008, 19:45"
1,Iris,Goo Goo Dolls,294000,02a75d1a-977b-4430-9db4-5d02d26e1d85,[],0,1095325,8483695,"09 Aug 2008, 10:22"
2,Everlong,Foo Fighters,334000,00779e5b-e581-3fcb-b0af-d6150e446b23,"[{'name': 'rock', 'url': 'https://www.last.fm/...",0,3083240,45555874,"22 Jul 2008, 14:03"
3,"Lover, You Should've Come Over",Jeff Buckley,403000,027cdcbb-96e7-351b-a7b5-d1004440e0f1,"[{'name': 'rock', 'url': 'https://www.last.fm/...",0,1588141,30534588,"24 Jan 2009, 15:20"
4,Still Into You,Paramore,216000,024d0cca-7f37-3456-9b42-1e56c4f0d460,"[{'name': 'rock', 'url': 'https://www.last.fm/...",0,1978453,26939991,"30 Apr 2013, 10:57"
...,...,...,...,...,...,...,...,...,...
34436,Hatch the Plan,Andy Stott,520000,2653b225-9f52-35cd-85c3-c30eac7a8405,[],0,43425,224503,NaN
34437,Jeanette,Kelly Lee Owens,374000,10c51eb3-39e6-4d59-af3a-eb20756ce53c,[],0,65338,300358,NaN
34438,A Thousand Nights,Gregor Tresher,289000,0175f1bd-b987-4abb-a7e5-3f88b0febd29,[],0,30766,115390,NaN
34439,Street Smarts,Narciss,393000,fcc5c4a4-1058-4b25-ac66-c56d6db883a6,[],0,305,974,NaN


In [ ]:
print(len(df_tracks_info))

NameError: name 'df_tracks_info' is not defined

### DUDA--<< Falta descargar data: hacer mas peticiones

* DUDA: que pasa con el pais? porque no aparecen?
* DUDA: otros datos que faltan? 
    * Endoint: tag.gettoptag (count y reach)
    * Endpoint: geo.??¿?¿ (¿?¿??)

# EDA

#### *Tipos de datos que contiene:*

| Tipo de variable                  | Ejemplos             | Qué analizar                         |
| --------------------------------- | -------------------- | ------------------------------------ |
| Numérica continua                 | duration, playcount  | medias, correlación                  |
| Numérica sesgada (muy común aquí) | playcount, listeners | log-transform, tests no paramétricos |
| Categórica                        | artista, género      | diferencias entre grupos             |
| Binaria                           | is_short_track       | comparación de grupos                |


#### *Data raw* 

In [5]:
df_raw = pd.read_csv('../data/raw/backup_tracks.csv')
df_raw.head()

,name,artist,duration,mbid,tag,streamable,listeners,playcount,published
0,The Chain,Fleetwood Mac,269000.0,021c73b9-c6d5-41de-a852-8b990b7e24e7,[],0.0,624167.0,4515515.0,"19 Dec 2008, 19:45"
1,Iris,Goo Goo Dolls,294000.0,02a75d1a-977b-4430-9db4-5d02d26e1d85,[],0.0,1095373.0,8483985.0,"09 Aug 2008, 10:22"
2,Everlong,Foo Fighters,334000.0,00779e5b-e581-3fcb-b0af-d6150e446b23,"[{'name': 'rock', 'url': 'https://www.last.fm/...",0.0,3084177.0,45580338.0,"22 Jul 2008, 14:03"
3,"Lover, You Should've Come Over",Jeff Buckley,403000.0,027cdcbb-96e7-351b-a7b5-d1004440e0f1,"[{'name': 'rock', 'url': 'https://www.last.fm/...",0.0,1589176.0,30569637.0,"24 Jan 2009, 15:20"
4,Still Into You,Paramore,216000.0,024d0cca-7f37-3456-9b42-1e56c4f0d460,"[{'name': 'rock', 'url': 'https://www.last.fm/...",0.0,1979430.0,26960812.0,"30 Apr 2013, 10:57"


In [47]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 34417 entries, 0 to 34416
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   name        34370 non-null  str    
 1   artist      34369 non-null  str    
 2   duration    34370 non-null  float64
 3   mbid        34348 non-null  str    
 4   tag         34370 non-null  str    
 5   streamable  34370 non-null  float64
 6   listeners   34370 non-null  float64
 7   playcount   34370 non-null  float64
 8   published   13653 non-null  str    
dtypes: float64(4), str(5)
memory usage: 2.4 MB


> **Observaciones:**  
> * Tenemos un total de xxx filas y xxx columnas:  
>   - Columnas numéricas: duration, playcount, listeners (variables clave)  
>   - Columnas string: name, image, country, streamable,genre_tag (etiquetas)  
>   - Columnas identificadoras: url, mbid (código identificador)
> * Valores nulos: Las filas que contienen más valores nulos son playcount, listeners y mbid. 




#### *Descripción de variables del dataset:* 



>**Variables principales:**
> 
> * **name** Nombre de la canción. Identificador principal del track.
> 
> * **artist** Nombre del artista. **Permite agrupar por creador y analizar popularidad.**
> 
> * **duration** Duración de la canción en milisegundos.**(Útil para analizar tendencias** (ej: canciones cortas tipo TikTok- ¿como? no tengo datos de tiktok))
> 
> * **playcount** Número total de reproducciones. (Indicador directo de popularidad global)
> 
> * **listeners** Número de usuarios únicos que han escuchado la canción. **(Permite medir alcance real (no solo repeticiones))**

---

> **Variables contextuales**
> 
> * **MISSING --> country** País desde donde se recoge el dato. (Permite análisis geográfico del mercado musical.)
> 
> * **tag** Género musical asociado (tag de Last.fm). (Permite segmentar por estilo musical.)  
>  * **published** Fecha de publicación --> get more info ¿Que siginifca realmetne?

---

> **Variables técnicas**
> 
> * **mbid** Identificador único de MusicBrainz. (Sirve para eliminar duplicados con precisión.)
> 
> * **streamable** Indica si la canción es reproducible. (No aporta valor para análisis de popularidad.)

#### *Data merged* 

* Se une la información de los diferentes CSVs recolentados hasta ahora para crear el nuevo DataSet. 

In [ ]:
import pandas as pd

# Cargar datos
df_lastfm_clean_ds = pd.read_csv('../data/raw/lastfm_dataset.csv')
df_backup = pd.read_csv('../data/raw/backup_tracks.csv')

# Nos quedamos solo con lo necesario
df_country = df_lastfm_clean_ds[['mbid', 'country']]

# Merge por mbid
df_merged = df_backup.merge(df_country, on='mbid', how='left')

# Ver resultado
df_merged.head()

In [ ]:
df_merged.info()

In [ ]:
df_merged.to_csv('../data/processed/df_merged-data.csv', index=False)

##### Información de merged data

In [ ]:
print('--- Info básica ---')
df_merged.info()

print('\n--- Distribución de country ---')
print(df_merged['country'].value_counts().head(15))

print('\n--- Distribución de genre_tag ---')
print(df_merged['genre_tag'].value_counts().head(15))

In [ ]:
df_merged = (df_clean.sort_values('playcount', ascending=False).head(15))
df_merged[['name', 'artist','tag']]

## Análisis de los datos 

### 1. Limpieza de datos

In [ ]:
df_clean = df_merged.copy()
df_clean.head()

,name,artist,duration,mbid,tag,streamable,listeners,playcount,published
0,The Chain,Fleetwood Mac,269000.0,021c73b9-c6d5-41de-a852-8b990b7e24e7,[],0.0,624167.0,4515515.0,"19 Dec 2008, 19:45"
1,Iris,Goo Goo Dolls,294000.0,02a75d1a-977b-4430-9db4-5d02d26e1d85,[],0.0,1095373.0,8483985.0,"09 Aug 2008, 10:22"
2,Everlong,Foo Fighters,334000.0,00779e5b-e581-3fcb-b0af-d6150e446b23,"[{'name': 'rock', 'url': 'https://www.last.fm/...",0.0,3084177.0,45580338.0,"22 Jul 2008, 14:03"
3,"Lover, You Should've Come Over",Jeff Buckley,403000.0,027cdcbb-96e7-351b-a7b5-d1004440e0f1,"[{'name': 'rock', 'url': 'https://www.last.fm/...",0.0,1589176.0,30569637.0,"24 Jan 2009, 15:20"
4,Still Into You,Paramore,216000.0,024d0cca-7f37-3456-9b42-1e56c4f0d460,"[{'name': 'rock', 'url': 'https://www.last.fm/...",0.0,1979430.0,26960812.0,"30 Apr 2013, 10:57"


* Eliminar tracks sin nombre ni artista (sin identidad no sirven)

In [ ]:
# df_clean.dropna(subset=['name', 'artist'])

* Data que no aporta información relevante

In [ ]:
((df_merged['country'] == 'UNKNOWN') | (df_merged['country'] == 'GLOBAL')).sum()

In [ ]:
df_clean[~df_clean['country'].isin(['UNKNOWN', 'GLOBAL'])] # DUDA tengo que dropear o ya estoy filtirando?

#### Columna de géneros 
* En la Api cada usuario escribe el género de la canción por eso en la columna 'tag' se encuentran listas de diccionarios, son los géneros en los que distintos usuarios han categorizado el track.

In [50]:
df_clean['tag'][2]

"[{'name': 'rock', 'url': 'https://www.last.fm/tag/rock'}, {'name': 'alternative rock', 'url': 'https://www.last.fm/tag/alternative+rock'}, {'name': '90s', 'url': 'https://www.last.fm/tag/90s'}, {'name': 'alternative', 'url': 'https://www.last.fm/tag/alternative'}, {'name': 'foo fighters', 'url': 'https://www.last.fm/tag/foo+fighters'}]"

In [51]:
df_clean['tag'].isna().sum()

np.int64(47)

> **Observaciones:** Hay 47 filas que no tienen información del género de la canción.

In [60]:
df_clean['tag'].apply(type).value_counts() # DUDA: NO ENTIENDO A QUE SE REFIERE CON 47 FILAS FLOAT, SON LISTAS VACIAS?

tag
<class 'str'>      34370
<class 'float'>       47
Name: count, dtype: int64

* Limpieza de la columna:

In [56]:
import ast
import numpy as np

def clean_tag(x):
    if pd.isna(x):
        return np.nan
    
    # Si es string, intentar convertir
    if isinstance(x, str):
        try:
            x = ast.literal_eval(x)
        except:
            return np.nan

    # Si es lista válida
    if isinstance(x, list) and len(x) > 0:
        if isinstance(x[0], dict):
            return x[0].get('name', np.nan)

    return np.nan

df_clean['tag_clean'] = df_clean['tag'].apply(clean_tag)

In [57]:
df_clean[['tag', 'tag_clean']].head(10)

,tag,tag_clean
0,[],NaN
1,[],NaN
2,"[{'name': 'rock', 'url': 'https://www.last.fm/...",rock
3,"[{'name': 'rock', 'url': 'https://www.last.fm/...",rock
4,"[{'name': 'rock', 'url': 'https://www.last.fm/...",rock
5,"[{'name': 'rock', 'url': 'https://www.last.fm/...",rock
6,"[{'name': 'rock', 'url': 'https://www.last.fm/...",rock
7,"[{'name': 'rock', 'url': 'https://www.last.fm/...",rock
8,"[{'name': 'rock', 'url': 'https://www.last.fm/...",rock
9,"[{'name': 'rock', 'url': 'https://www.last.fm/...",rock


In [59]:
df_clean['tag_clean'].nunique()

312

In [80]:
df_clean['tag'] = df_clean['tag_clean']

> **Observaciones:** Después del alimpieza de la columna se encuentran 312 géneros distintos en todo el DataSet

<!-- ### 1. Top canciones y artistas por periodo <a id='top'></a>

**Objetivo:** identificar qué canciones y artistas dominan el mercado global.  
**Datos:** `chart.getTopTracks` + `chart.getTopArtists` -->

#### Columna de la información de publicación 

In [63]:
df_clean['published_date'] = pd.to_datetime(df_clean['published'],format='%d %b %Y, %H:%M',errors='coerce').dt.date

Explicacion del codigo:

In [65]:
pd.DataFrame(df_clean['published_date'])

,published_date
0,2008-12-19
1,2008-08-09
2,2008-07-22
3,2009-01-24
4,2013-04-30
...,...
34412,NaT
34413,NaT
34414,NaT
34415,NaT


In [66]:
df_clean['published_date'].isna().sum()

np.int64(20764)

> **Observaciones:** Más de la mitad de los datos no tienen fecha de publicación. 

## Feature Engineering 



### Creacion de futures

Se crean variables derivadas que usaremos para fortalecer los análisis y resultados del EDA y los modelos ML.

* Duración en minutos: Last.fm devuelve duración en milisegundos no en minutos, se convierten los datos a minutos:

In [ ]:
df_clean['duration_min'] = df_clean['duration'] / 60000
pd.DataFrame(df_clean['duration_min'].apply(lambda x: f"{x:.2f}"))



* Flag canción corta: <2.5 min = formato TikTok/Reels

In [ ]:
df_clean['is_short_track'] = (df_clean['duration_min'] < 2.5).astype(int)


* Flag canción viral: ¿Qué hace que una canción sea hit?

In [ ]:
threshold = df_clean['playcount'].quantile(0.90)

df_clean['is_hit'] = (df_clean['playcount'] >= threshold).astype(int)


* Engagement: cuántas veces escucha la canción cada oyente único:
    * Ratio alto → canción que engancha y se repite mucho

In [ ]:

df_clean['playcount_per_listener'] = df_clean['playcount'] / df_clean['listeners'].replace(0, np.nan)


* Log-transform de playcount y listeners

df_clean['log_playcount'] = np.log1p(df_clean['playcount'])
df_clean['log_listeners'] = np.log1p(df_clean['listeners'])

* Popularidad derivada --> DUDA: a que se refiere?

In [ ]:
df_clean['popularity_ratio'] = (
    df_clean['playcount'] / df_clean['playcount'].sum()
)

* Ratio de descurbimiento: canciones muy repetidas vs canciones “descubiertas pero no repetidas”

In [ ]:
df_clean['listener_to_play_ratio'] = (
    df_clean['listeners'] / df_clean['playcount'].replace(0, np.nan)
)

* Features de artista (métricas agregadas)

In [ ]:
artist_stats = df.groupby('artist').agg(
    artist_track_count     = ('name', 'count'),
    artist_total_playcount = ('playcount', 'sum'),
).reset_index()
# df = df.merge(artist_stats, on='artist', how='left')


* Peso del track en el catálogo del artista

In [ ]:
df['track_share_of_artist'] = df['playcount'] / df['artist_total_playcount'].replace(0, np.nan)

# df_clean = df.reset_index(drop=True)

* Peso de este track en el catálogo del artista

In [ ]:
df['peso_en_artista'] = df['playcount'] / df['plays_totales_artista'].replace(0, np.nan)

### Análisis de las nuevas features


#### Top artistas: volumen vs presencia


In [ ]:
top_artists = (df_clean.groupby('artist')[['playcount', 'name']].agg({'playcount': 'sum', 'name': 'count'}).rename(columns={'playcount': 'total_plays', 'name': 'n_tracks'}).sort_values('total_plays', ascending=False).head(15))


##### Gráficos:


In [ ]:
top_artists.sort_values('total_plays').plot.barh(y='total_plays', figsize=(8,5), title='Top artistas por reproducciones')
top_artists.sort_values('n_tracks').plot.barh(y='n_tracks', figsize=(8,5), title='Top artistas por nº de tracks')

plt.show()



* Concentración de reproducciones en top 5 artistas


In [ ]:
top5_share = top_artists['total_plays'].head(5).sum() / df_clean['playcount'].sum() * 100
print(f"Top 5 concentran {top5_share:.1f}% de reproducciones")

* Top 15 canciones por playcount:

#### * Features de artista (métricas agregadas)

DUDA: Revisa función .agg()

In [ ]:
artist_stats = df.groupby('artist').agg(
    artist_track_count     = ('name', 'count'),
    artist_total_playcount = ('playcount', 'sum'),
).reset_index()
df = df.merge(artist_stats, on='artist', how='left')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].barh(top_artistas['artist'], top_artistas['plays_totales'] / 1e6,
             color=sns.color_palette('Purples_r', 15))
axes[0].invert_yaxis()
axes[0].set_xlabel('Plays totales (millones)')
axes[0].set_title('Top 15 Artistas por Reproducciones')

axes[1].barh(top_artistas['artist'], top_artistas['n_tracks'],
             color=sns.color_palette('Oranges_r', 15))
axes[1].invert_yaxis()
axes[1].set_xlabel('Nº de tracks en el dataset')
axes[1].set_title('Top 15 Artistas por Presencia')

plt.suptitle('🎤 Artistas dominantes', fontweight='bold')
plt.tight_layout()
plt.show()

# Concentración del mercado
top5_share = top_artistas.head(5)['plays_totales'].sum() / df_clean['playcount'].sum() * 100
print(f'Los top 5 artistas concentran el {top5_share:.0f}% de reproducciones totales.')
print('→ Mercado muy concentrado. Para artistas nuevos, el nicho puede ser más efectivo.')

## Análisis estadísticos

### Análisis del mercado musical

In [ ]:
df_mercmusic = df_clean.copy()

#### Correlaciones:

* Se utiliza el método Spearman porque los datos tienden a estar sesgados (DUDA que es long tail?)

In [ ]:
from scipy.stats import spearmanr

spearmanr(df_clean['duration_min'], df_clean['playcount'])

> Observaciones:
* ¿Las canciones más cortas tienen más plays?
* ¿Más listeners → más engagement?

**Heatmap**

In [ ]:

cols_numericas = [
    'playcount_log', 'listeners_log', 'engagement',
    'duration_min', 'is_short_track',
    'n_tracks_artista', 'peso_en_artista'
]
cols_numericas = [c for c in cols_numericas if c in df_clean.columns]

corr = df_clean[cols_numericas].corr()

fig, ax = plt.subplots(figsize=(10, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))  # solo triángulo inferior
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, linewidths=0.5, ax=ax)
ax.set_title('Correlaciones entre variables', fontweight='bold')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

# Variables más correlacionadas con playcount
corr_target = corr['playcount_log'].drop('playcount_log').abs().sort_values(ascending=False)
print('Variables más relacionadas con la popularidad (playcount_log):')
for feat, r in corr_target.head(4).items():
    signo = '+' if corr.loc['playcount_log', feat] > 0 else '-'
    print(f'  {signo} {feat}: r={r:.3f}')

#### Comparación entre grupos

* ¿Las canciones cortas tienen más éxito?
* ¿Porque? ejemplo : “Las canciones cortas funcionan mejor en plataformas tipo TikTok”

In [ ]:
from scipy.stats import mannwhitneyu

short = df_clean[df_clean['is_short_track'] == 1]['playcount']
long = df_clean[df_clean['is_short_track'] == 0]['playcount']

mannwhitneyu(short, long)

* Duración vs popularidad : ¿Las canciones cortas son más populares?


In [ ]:
rangos = pd.cut(
    df_clean['duration_min'],
    bins=[0, 1.5, 2.5, 3.5, 4.5, 6, 100],
    labels=['<1.5m', '1.5-2.5m', '2.5-3.5m', '3.5-4.5m', '4.5-6m', '>6m']
)
pop_por_rango = df_clean.groupby(rangos, observed=True)['playcount_log'].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

pop_por_rango.plot.bar(ax=axes[0], color=sns.color_palette('RdYlGn', 6))
axes[0].set_title('Popularidad media por duración')
axes[0].set_ylabel('log(Playcount) medio')
axes[0].tick_params(axis='x', rotation=30)

df_clean.boxplot(column='playcount_log', by='is_short_track', ax=axes[1])
axes[1].set_title('Corta (<2.5m) vs Larga')
axes[1].set_xlabel('is_short_track (0=No, 1=Sí)')
axes[1].set_ylabel('log(Playcount)')
plt.suptitle('')

plt.suptitle('⏱️ ¿Afecta la duración a la popularidad?', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

corta = df_clean[df_clean['is_short_track'] == 1]['playcount_log'].mean()
larga = df_clean[df_clean['is_short_track'] == 0]['playcount_log'].mean()
ganador = 'cortas (<2.5 min)' if corta > larga else 'largas (≥2.5 min)'
print(f'Las canciones {ganador} tienen mayor popularidad media.')
print(f'  Cortas: {corta:.3f} | Largas: {larga:.3f}')

# ML


**¿Qué predecimos?**
- **Clasificación:** ¿es esta canción un hit? (playcount ≥ percentil 75)
- **Regresión:** ¿qué playcount tendrá? (predecimos el valor continuo)

**Variables disponibles para predecir:**

| Variable | Tipo | Valor para ML |
|---|---|---|
| `listeners_log` | numérica | ⭐⭐⭐ Mejor predictor |
| `genre_tag` | categórica | ⭐⭐ El género importa mucho |
| `country` | categórica | ⭐⭐ Diferencias culturales reales |
| `duration_min` | numérica | ⭐ La duración tiene algo de señal |
| `is_short_track` | binaria | ⭐ Formato TikTok |
| `n_tracks_artista` | numérica | ⭐ Artistas con más tracks = más conocidos |
| `peso_en_artista` | numérica | ⭐ ¿Es el hit del artista? |

**Limitación importante:** Last.fm **no tiene** danceability, energy, valence, BPM.  
Para esas features necesitarías Spotify API. Con lo que tenemos, el modelo predice bien la popularidad general pero no las características acústicas de un hit.

### Preparar datos para ML --< DUDA qué hace el codigo>

* Target: is_hit (1 = hit, 0 = no hit)
* Umbral: percentil 75 de playcount

In [ ]:

umbral_hit = df_clean['playcount'].quantile(0.75)
df_clean['is_hit'] = (df_clean['playcount'] >= umbral_hit).astype(int)

print(f'Umbral de hit: {umbral_hit:,.0f} reproducciones (percentil 75)')
print(f'Hits:     {df_clean["is_hit"].sum():,} ({df_clean["is_hit"].mean()*100:.0f}%)')
print(f'No hits:  {(1-df_clean["is_hit"]).sum():,} ({(1-df_clean["is_hit"].mean())*100:.0f}%)')

# Encodear variables categóricas (texto → número)
df_ml = df_clean.copy()
le_genero = LabelEncoder()
le_pais   = LabelEncoder()

df_ml['genre_encoded']   = le_genero.fit_transform(df_ml['genre_tag'].fillna('UNKNOWN'))
df_ml['country_encoded'] = le_pais.fit_transform(df_ml['country'].fillna('UNKNOWN'))

# Features: qué columnas usamos para predecir
FEATURES = [
    'listeners_log',
    'duration_min',
    'is_short_track',
    'genre_encoded',
    'country_encoded',
    'n_tracks_artista',
    'peso_en_artista',
    'engagement',
]
FEATURES = [f for f in FEATURES if f in df_ml.columns]

# Eliminar filas con NaN en las features
df_ml_ok = df_ml[FEATURES + ['playcount_log', 'is_hit']].dropna()
print(f'\nDataset para ML: {len(df_ml_ok):,} filas | {len(FEATURES)} features')
print(f'Features: {FEATURES}')

### Imports

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, r2_score, mean_absolute_error


In [ ]:
X = df_ml_ok[FEATURES]
y_clf = df_ml_ok['is_hit']        # para clasificación
y_reg = df_ml_ok['playcount_log'] # para regresión

X_train, X_test, y_clf_train, y_clf_test = train_test_split(
    X, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)
_, _, y_reg_train, y_reg_test = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)

print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')

### **Regresión lineal y/o logarítmica**

*  Qué variables impactan de verdad?
* ¿Cuánto influyen?

In [ ]:
import statsmodels.api as sm
import numpy as np

df_model = df_clean.copy()

# log porque playcount suele ser heavy-tailed
df_model['log_playcount'] = np.log1p(df_model['playcount'])

X = df_model[['duration_min', 'playcount_per_listener']]
X = sm.add_constant(X)

y = df_model['log_playcount']

model = sm.OLS(y, X).fit()
print(model.summary())

* **Random Forest**

In [ ]:
rf_reg = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_reg.fit(X_train, y_reg_train)
y_pred_reg = rf_reg.predict(X_test)

print(f'--- Regresión (predecir log de plays) ---')
print(f'R²:  {r2_score(y_reg_test, y_pred_reg):.3f}  (1.0 = perfecto)')
print(f'MAE: {mean_absolute_error(y_reg_test, y_pred_reg):.3f}')

### **Clasificación**

* **ANOVA / Kruskal-Wallis**: ¿Hay diferencias entre géneros?

In [ ]:
from scipy.stats import kruskal

groups = [group['playcount'].values for name, group in df_clean.groupby('genre')]

kruskal(*groups)

* **XGBOOST**

In [ ]:
try:
    from xgboost import XGBClassifier
    XGBOOST = True
    print('✅ XGBoost disponible')
except ImportError:
    XGBOOST = False
    print('⚠️  XGBoost no instalado: pip install xgboost')

print('✅ Librerías ML cargadas')

* **Random Forest**

In [ ]:
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_clf.fit(X_train, y_clf_train)
y_pred_clf = rf_clf.predict(X_test)

### **Feature Importance: ¿qué determina si una canción es un hit?**

In [ ]:


importancias = pd.Series(rf_clf.feature_importances_, index=FEATURES).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))
importancias.plot.bar(ax=ax, color=sns.color_palette('Blues_r', len(importancias)))
ax.set_title('🎯 ¿Qué determina si una canción es un hit?', fontweight='bold')
ax.set_ylabel('Importancia relativa')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

print('Las 3 variables más importantes:')
for feat, val in importancias.head(3).items():
    print(f'  → {feat}: {val*100:.1f}%')

# Función de predicción final Streamlit <a id='6'></a>

Esta función es el **output de negocio** del módulo:  
> "Tu canción tiene X% de probabilidad de ser un hit"

**Para usar en Streamlit:** esta función puede llamarse directamente desde la app.

In [ ]:
def predecir_hit(nombre, artista, duracion_min, genero, pais, oyentes_estimados):
    """
    Predice la probabilidad de que una canción sea un hit.
    
    Parámetros:
        nombre            — nombre de la canción
        artista           — nombre del artista  
        duracion_min      — duración en minutos (ej: 3.5)
        genero            — género musical (ej: 'pop', 'rock')
        pais              — país de lanzamiento (ej: 'spain')
        oyentes_estimados — estimación inicial de oyentes únicos
    
    Devuelve: probabilidad de hit (0-100%)
    """
    # Encodear género y país (si no están en el vocabulario del encoder, usamos 0)
    if genero in le_genero.classes_:
        genre_enc = le_genero.transform([genero])[0]
    else:
        genre_enc = 0

    if pais in le_pais.classes_:
        country_enc = le_pais.transform([pais])[0]
    else:
        country_enc = 0

    # Crear el vector de features
    datos = pd.DataFrame([{
        'listeners_log'  : np.log1p(oyentes_estimados),
        'duration_min'   : duracion_min,
        'is_short_track' : int(duracion_min < 2.5),
        'genre_encoded'  : genre_enc,
        'country_encoded': country_enc,
        'n_tracks_artista': 1,    # artista nuevo = 1 track
        'peso_en_artista': 1.0,   # único track del artista = 100%
        'engagement'     : 5.0,   # engagement inicial estimado
    }])
    datos = datos[FEATURES]  # mismo orden que el modelo espera

    probabilidad = rf_clf.predict_proba(datos)[0][1] * 100

    if probabilidad >= 70:
        clasificacion = '🚀 Hit potencial'
    elif probabilidad >= 45:
        clasificacion = '🟡 Potencial medio'
    else:
        clasificacion = '📉 Bajo potencial'

    print('='*50)
    print(f'  🎵 {nombre} — {artista}')
    print('='*50)
    print(f'  Probabilidad de hit:  {probabilidad:.1f}%')
    print(f'  Clasificación:        {clasificacion}')
    print(f'  Género:               {genero}')
    print(f'  País:                 {pais}')
    print(f'  Duración:             {duracion_min:.1f} min', end='')
    print(f' (formato corto ⏱️)' if duracion_min < 2.5 else '')
    print('='*50)

    return probabilidad


# --- Ejemplo de uso ---
predecir_hit(
    nombre='Mi Canción',
    artista='Mi Artista',
    duracion_min=2.3,
    genero='pop',
    pais='spain',
    oyentes_estimados=50000
)


## 💡 Ideas para Streamlit / App web



Con lo que tienes, puedes construir una app con estas secciones:



### Página 1 — Predictor de hit


```python
# En app.py:
import streamlit as st
import pickle


* Cargar modelo y encoders guardados

In [ ]:


rf_clf = pickle.load(open('modelo_hits.pkl', 'rb'))
le_genero = pickle.load(open('le_genero.pkl', 'rb'))

st.title('🎵 ¿Será un hit?')
duracion   = st.slider('Duración (min)', 1.0, 8.0, 3.5)
genero     = st.selectbox('Género', ['pop', 'rock', 'hip-hop', 'electronic'])
oyentes    = st.number_input('Oyentes estimados', value=10000)

if st.button('Predecir'):
    prob = predecir_hit('Mi canción', 'Mi artista', duracion, genero, 'spain', oyentes)
    st.metric('Probabilidad de hit', f'{prob:.0f}%')
    st.progress(prob / 100)



### Página 2 — Dashboard del mercado


- Top 15 canciones (barplot)
- Géneros por popularidad (barplot)
- Mapa de calor géneros × países



### Página 3 — Explorador de artistas


- Input: nombre del artista
- Output: sus tracks en el dataset, popularidad media, géneros



### Cómo guardar y cargar el modelo


In [ ]:
import pickle

# Guardar (en el notebook)
pickle.dump(rf_clf,    open('models/modelo_hits.pkl', 'wb'))
pickle.dump(le_genero, open('models/le_genero.pkl', 'wb'))
pickle.dump(le_pais,   open('models/le_pais.pkl', 'wb'))

# Cargar (en Streamlit)
rf_clf    = pickle.load(open('models/modelo_hits.pkl', 'rb'))
le_genero = pickle.load(open('models/le_genero.pkl', 'rb'))



### Ideas de ML adicionales con las variables actuales



| Idea | Variables necesarias | Factibilidad |
|---|---|---|
| Predecir si un track será hit | listeners, genre, country, duration | ✅ Ya tienes esto |
| Clustering de géneros musicales | playcount, engagement, pct_cortas | ✅ KMeans con tus datos |
| Ranking de países con más potencial | plays_medio por país | ✅ Con geo.getTopTracks |
| Detectar artistas emergentes | n_tracks_artista, plays_medio, tendencia | ✅ Con datos temporales |
| Predecir viralidad por duración | is_short_track + playcount | ✅ Ya tienes esto |
| Audio features (danceability, BPM...) | — | ❌ Necesitas Spotify API |



### Para el backup_tracks.csv (tiene 'published' = fechas reales)


El archivo `backup_tracks.csv` tiene un campo `published` con fechas reales de publicación.  
Esto permite análisis temporales reales: qué géneros triunfan en verano, tendencias por año, etc.


# Guardado de modelos

##  Guardar modelos para usar en Streamlit 


In [ ]:
import pickle
import os

os.makedirs('models', exist_ok=True)

pickle.dump(rf_clf,    open('models/modelo_hits_clf.pkl', 'wb'))
pickle.dump(rf_reg,    open('models/modelo_plays_reg.pkl', 'wb'))
pickle.dump(le_genero, open('models/le_genero.pkl', 'wb'))
pickle.dump(le_pais,   open('models/le_pais.pkl', 'wb'))



## Guardar también las features usadas (importante: el orden debe coincidir)


In [ ]:
with open('models/features.txt', 'w') as f:
    f.write('\n'.join(FEATURES))

print('✅ Modelos guardados en /models/')
print('   Para cargarlos en Streamlit:')
print('   rf_clf = pickle.load(open("models/modelo_hits_clf.pkl", "rb"))')